# ETE Basics

---

**Quick reminder**:

- To install ETE: `pip install ete4` (see https://github.com/etetoolkit/ete)
    - If you want the very latest: `pip install --force-reinstall https://github.com/etetoolkit/ete/archive/ete4.zip`
- Documentation: https://etetoolkit.github.io/ete/

---

## Working with the Tree structure

In general, a *tree* is a [set of linked nodes](https://en.wikipedia.org/wiki/Tree_(data_structure)) that simulates a hierarchical [tree structure](https://en.wikipedia.org/wiki/Tree_structure).

Conceptually, they look like this:

![tree example](330px-Tree_%28computer_science%29.svg.png)
<!-- from https://upload.wikimedia.org/wikipedia/commons/thumb/5/5f/Tree_%28computer_science%29.svg/330px-Tree_%28computer_science%29.svg.png -->

ETE provides a class `Tree` to work with them. From our Python console, we can import it from the `ete4` module:

In [ ]:
from ete4 import Tree

The `Tree` class can be used to construct trees. We can create a single node by calling `Tree()` without any arguments:

In [ ]:
# Empty tree (single node).
t = Tree()

Or we can call it with a dictionary specifying the properties of that single node.

In [ ]:
# Also a single node, but with some properties.
t = Tree({'name': 'root', 'dist': 1.0, 'support': 0.5, 'coolness': 'high'})

The properties of a node are stored in its `props` dictionary. With the previous example, writing `t.props` will show us a dictionary that should look familiar.

In [ ]:
t.props  # where the properties of a node are stored

Some of the properties belong to the *node* itself (like `name`), while others belong to the *branch* that links it to its parent node (like `dist` and `support`). Since each node has one branch that connects it to its parent (except for the root node), we can keep together its *node properties* and *branch properties*.

The trees we are mainly interested in are [phylogenetic trees](https://en.wikipedia.org/wiki/Phylogenetic_tree), which show the relationships among various biological species. Their branches will normally have a length (`dist`) that measures in some form the evolutionary distance between the nodes, and a `support` that measures the confidence that we have in that connection. But ETE's `Tree` class is very general and can be used to work with any kind of trees.

Before going further, let's look at a simple example of tree with ETE. We are going to use the `populate()` method to populate a tree with a random topology.

As with any function, we can check its documentation with `help()`:

In [ ]:
help(t.populate)

Let's populate `t` with a random topology with 10 leaves:

In [ ]:
t.populate(10)

Now that our tree `t` is more interesting, let's write `print(t)` to see a simple visualization of the tree itself:

In [ ]:
print(t)

Soon we will see more convenient ways of creating and visualizing trees.

<div style="background-color:lightblue; padding:1em">

  ### Exercise: Creating a simple tree (random)

Make sure you have ete4 installed and:

- Create a tree with 5 leaves, whose branches have random lengths
    - Remember that Python's `random` module has helpful functions, like `random()`
- We can access the first node as `t.children[0]` (more on that later). What are its properties?
- Print a text representation of the tree
</div>

In [ ]:
# Solution.

import random

t = Tree()

t.populate(5, dist_fn=random.random)

print('Properties of first node:', t.children[0].props)

print('Text representation of the tree:')
print(t)

### Basic attributes

Every tree node is represented by the `Tree` structure. In addition to its properties (stored in the `props` dictionary), it has a list of its `children` (which can be viewed as trees themselves), and a pointer to its parent node (`up`), which is a bigger tree, or `None` in the case of the root node.

![tree](tree_parts.png)
<!-- from https://gitlab.com/jordibc/tree-explorer/-/raw/ete-like/docs/img/tree_parts.png -->

If `node` is a tree node (a `Tree` instance), we can always access its properties with `node.props`. For example, `node.props['coolness']` if that property was defined.

We mentioned that **there are three basic properties often present in phylogenetic trees**. ETE provides a shorter way to access them:

| Attribute        | Description |
| :----------      | :---------: |
| **node.dist**    | Distance from the node to its parent (branch length) |
| **node.support** | Reliability of the partition defined by the node (like bootstrap support) |
| **node.name**    | Name of the node |

ETE represents a node with its name if it has it and a unique internal hash that identifies it. We will see that representation in some of the outputs when we work with trees. For example, if we try to see the `children` from `t` itself:

In [ ]:
print(t.children)

So we see that `t` has 2 children nodes, that are unnamed.

What happens if we see the first child?

In [ ]:
print(t.children[0])

We didn't get that "<Tree at 0x....>" representation anymore! That's because ETE is using the text visual representation for the tree. Also, note that `t.children[0]` is a tree itself: the subtree hanging from the first branch of `t`. This agrees with what we said before about how we can see nodes as being trees themselves.

To get its object representation we can, as with any Python structure, use `repr()` or `%r`:

In [ ]:
print('The representation of t.children[0] is:', repr(t.children[0]))  # with repr()

print('The representation of t.children[0] is "%r"' % t.children[0])  # with %r

We also get the object representation if we write the tree on the Python console, without `print()`ing it:

In [ ]:
t

What about `t`'s parent node (the next one up in the tree hierarchy)?

In [ ]:
print('t.up = %r' % t.up)

It has none. No hashes, no "Tree", just Python's `None`. ETE sets a parent node to `None` to indicates that it is the absolute *root* of the whole tree and has no parent.

Another way of saying the same is:

In [ ]:
print('t is the root of the tree:', t.is_root)

<div style="background-color:lightblue; padding:1em">

  ### Exercise: Creating a simple tree (2 leaves)

- Create two nodes with names "a" and "b"
- Create another node "root" with them as children
    - Have a look at the documentation of `Tree()`, or use `root.add_child()`
    - If we wanted to manually append them to `root.children`, what would happen?
- Print a text representation of the resulting tree
</div>

We can make full trees by creating individual nodes with `node = Tree()` and then passing them when constructing other trees as their childs with `Tree({...}, children=[node1, node2, ...])`. Or alternatively by calling the `add_child()` or `add_children()` members (see [ETE's documentation for that](https://etetoolkit.github.io/ete/tutorial/tutorial_trees.html#creating-trees-from-scratch)). But for most cases, that would be tedious.

Let's see other ways.

## Loading and writing trees, the Newick format and parsers

The [Newick format](https://en.wikipedia.org/wiki/Newick_format) is one of the most widely used standard representations of trees in bioinformatics. It uses nested parentheses to represent hierarchical data structures as text strings.

### Reading newicks

We can construct a tree from its newick representation by passing it as a string to `Tree()`.

In [ ]:
newick = '((((A,B),C),D),(E,F));'

# Load a tree structure from a newick string. It returns the root node.
t = Tree(newick)

How does it look like?

In [ ]:
print(t)

Other than passing a string with the newick representation, we can pass instead a file object that contains the newick string.

In [ ]:
# Load a tree structure from a newick file.
t = Tree(open('tree.nw'))

In [ ]:
print('Loaded tree:')
print(t)

In [ ]:
print('Initialized with file containing:', open('tree.nw').read())

### Alternative parsings

The original newick standard is able to encode information about the tree topology, branch distances and node names. Nevertheless, it is common to find some variations of the format.

In particular, one often finds trees like `(A,B)0.3,...` where the "A" and "B" are names, but the 0.3 is a *support value* instead of the *name* of an internal node. But one can also find trees like `(A,B)C,...`.

Since in phylogenetics the most common format has support values for internal nodes, ETE uses that as default. But it also provides a flexible way to parse almost any kind of format. To use it, it accepts a `parser` as an argument.

The parser can be quite sophisticated, but for our purposes we will only use two shortcuts: `parser=0` (or `parser='support'`) for internal nodes with support (ETE's default), and `parser=1` (or `parser='name'`) for internal nodes with names.

We can find more about formats in [ETE's documentation](https://etetoolkit.github.io/ete/tutorial/tutorial_trees.html#reading-and-writing-newick-trees).

In [ ]:
# We can specify how to parse the newick. For instance, internal nodes by default are
# interpreted as support values (parser=0), but we can interpret them as names with parser=1.
t = Tree('(A:1,(B:1,(C:1,D:1)E:0.5)F:0.5);', parser=1)

If we now see it with `print()`:

In [ ]:
print(t)

The names of the internal nodes are missing! That's because ETE shows by default the names of the leaf nodes only.

We can change that:

In [ ]:
print(t.to_str())

There they are. Using `to_str()` also got us more information. We will later see in more detail how to do text visualizations of trees.

### Writing newicks

Any `Tree` can be exported to newick using the `write()` method. When using it, we can also select which parser to use. We could even use this to convert between newick formats.

In [ ]:
# Load a tree with internal names.
t = Tree('(A:1,(B:1,(C:1,D:1)E:0.5)F:0.5);', parser=1)

In [ ]:
# Get its newick using the default parser.
t.write()

The names of the internal nodes are gone! Is it clear why?

In [ ]:
# To get the internal names we can change the parser.
t.write(parser=1)

With `parser=1` the names are back when converting to a newick.

We know where names, distances and support appear in the newick. But what about other properties? Let's find out.

In [ ]:
t.children[0].props['usefulness'] = 'not much'

In [ ]:
t.write()

No trace of the internal node names as expected (because we didn't use `parser=1`), but also no trace of the property we just added!

This is because ETE will output by default the minimal information required in the selected format. But we can ask to add other properties, which will be written in the so-called *extended Newick format*:

In [ ]:
t.write(props=['usefulness'])

There it is. And if we set it to the special value `props=None`, it will store all the properties:

In [ ]:
t.write(props=None)

This time we see also the "name" property stored for internal nodes, but as an extended attribute, since the default parser (`parser=0`) does not store them otherwise.

Finally, we can use `write()` to write into a file too:

In [ ]:
t.write(parser=1, outfile='new_tree.nw')

In [ ]:
print('The contents of file new_tree.nw are:')
print(open('new_tree.nw').read())

<div style="background-color:lightblue; padding:1em">

  ### Exercise: Tree from newick

For the following newick: `'(((a,b)ab,(c,d)cd:2)abcd,((e,f)ef,g)efg);'`

- Create a Tree from it. Which parser does it seem we should use?
- Change in that Tree the first branch name from `'abcd'` to `'ABCD'`
- Create a newick representation that shows the new name included
</div>

## Text visualization

Let's start with a simple tree:

In [ ]:
t = Tree('((((A:1,B:2)G:3,C:1)H:0.5,D:1)I:0.5,(E:1,F:2)J:1)root;', parser=1)
print(t)

We only see the names of the leaf nodes, but we can get more information with `to_str()`:

In [ ]:
print(t.to_str())

This shows all the properties and their values for all the nodes. We can instead select the properties we want to see and have a more compact representation:

In [ ]:
print(t.to_str(props=['name', 'dist'], compact=True))

The `props=...` argument will choose which properties to show and in which order. Note how we didn't set a dist for the root, and ETE uses the character ⊗ to mark its absence.

We can find the meaning of the arguments for `to_str()` by checking [the online documentation](https://etetoolkit.github.io/ete/reference/reference_tree.html#ete4.core.tree.Tree.to_str) or simply by writing `help(t.to_str)`.

---

## Exploring trees

Let's continue with our tree:

In [ ]:
print(t.to_str(props=['name'], compact=True))

We could get the node "A" by noticing its relationship with `t` and going over the children of the children of the children of the... until getting there. As in:

In [ ]:
t.children[0].children[0].children[0].children[0]  # seriously?

That's not much fun.

ETE provides several ways of getting to the same node that are more convenient. One possibility is to get there by passing the tuple `(0, 0, ...)` or whatever position we want, to the tree with the `t[tuple]` syntax. As in:

In [ ]:
t[0,0,0,0]  # better

### Find by name

A similar syntax can be used to get a node by its name, by using `t[name]`:

In [ ]:
t['A']

If there is more than one node named "A", `t['A']` will return the first one that ETE finds while traversing the tree.

Let's look at this node:

In [ ]:
A = t['A']

print('A.children =', A.children)
print('A.up = %r' % A.up)

Now we see the opposite to what happened with the root: `A` has no children, but has a parent. This is because `A` is a *leaf node*:

In [ ]:
print('t is the root node?', t.is_root, '  t is a leaf node?', t.is_leaf)
print('A is the root node?', A.is_root, '  A is a leaf node?', A.is_leaf)

We mentioned before that all nodes can be seen as trees themselves. How does it look like for `A`'s parent?

In [ ]:
print(A.up.to_str(props=['name']))

This makes sense: if we see the tree from the perspective of `A`'s parent, it is a tree that starts at `G` and there are only two more nodes (`A` and `B`).

### Find by common ancestry

Another powerful method of finding nodes is by **common ancestry** of a set of nodes. For example, let's say we want to get the last common ancestor of nodes `B` and `C`. First, here is `t` again:

In [ ]:
print(t.to_str(props=['name'], compact=True))

We can use ETE's `common_ancestors()` member function for that:

In [ ]:
ancestor_B_C = t.common_ancestor('B', 'C')
print(ancestor_B_C)

The node we called `ancestor_B_C` indeed contains `B` and `C`. We can in principle select any nodes in the tree with the functions that we have seen, but ETE also provides other ways that are sometimes more convenient.

By the way, did you notice that we passed strings like `'B'` and `'C'` to `t.common_ancestor()`? Should it not be node objects like `t['B']`? Since we called a method that belongs to `t`, ETE knows what we mean: the node in `t` with that name. This is the same, but often more convenient, than writing `t.common_ancestor(t['B'], t['C'])`. And ETE does it too in all methods that accept nodes as arguments.

### Find by properties

Let's add a "color" property to the leaves:

In [ ]:
t['A'].props['color'] = 'red'
t['B'].props['color'] = 'green'
t['C'].props['color'] = 'blue'
t['D'].props['color'] = 'green'
t['E'].props['color'] = 'blue'
t['F'].props['color'] = 'blue'

All the leaves are now colored (have been assigned the "color" property). Let's see them:

In [ ]:
print(t.to_str(props=['name', 'color'], show_internal=False, compact=True))

Now, if we want to get all the blue nodes, we could [find them by their properties](https://etetoolkit.github.io/ete/tutorial/tutorial_trees.html#finding-nodes-by-their-properties) with the `search_nodes()` method like:

In [ ]:
print('Nodes colored blue:')
for node in t.search_nodes(color='blue'):
    print('  -', node.name)

<div style="background-color:lightblue; padding:1em">

  ### Exercise: Investigating nodes
  
For the following newick: `'(((a,b)ab,(c,d)cd:2)abcd,((e,f)ef,g)efg)root;'`

- Get the last common ancestor of nodes a, c and d
- Is it the root node? Is it a leaf? Check with the corresponding attributes
- Is it the same as the the node named abcd?
</div>

For more flexible ways of getting nodes, we can simply traverse the tree and put conditions on the current node to see if we select it.

---

## Traversing trees: traverse, ancestors, descendants, pre/postorder

Often, when processing trees, all nodes need to be visited. This is called *tree traversing*.

There are different ways to traverse a tree structure depending on the order in which children nodes are visited. ETE implements the three most common strategies: *levelorder* ([breadth-first](https://en.wikipedia.org/wiki/Breadth-first_search)), *preorder*, and *postorder* (both [depth-first](https://en.wikipedia.org/wiki/Depth-first_search)).

![levelorder](330px-Sorted_binary_tree_breadth-first_traversal.svg.png)
<!-- from https://upload.wikimedia.org/wikipedia/commons/thumb/d/d1/Sorted_binary_tree_breadth-first_traversal.svg/330px-Sorted_binary_tree_breadth-first_traversal.svg.png -->

**levelorder** (each level at a time: F, B, G, A, D, I, C ,E ,H)

![levelorder2](440px-Sorted_binary_tree_ALL_RGB.svg.png)
<!-- from https://upload.wikimedia.org/wikipedia/commons/thumb/7/75/Sorted_binary_tree_ALL_RGB.svg/440px-Sorted_binary_tree_ALL_RGB.svg.png -->

**preorder** (in order of touching the <span style="color:red">red dots</span>: F, B, A, D, C, E, G, I, H)

**postorder** (in order of touching the <span style="color:blue">blue dots</span>: A, C, E, D, B, H, I, G, F)

To traverse a tree, we can use the method `traverse()`, which provides an iterator over all the nodes: 

In [ ]:
t = Tree('((A,(C,E)D)B,((H)I)G)F;', parser=1)

print(t.to_str(props=['name']))

print('Nodes in levelorder:', ', '.join(node.name for node in t.traverse('levelorder')))
print('Nodes in preorder:  ', ', '.join(node.name for node in t.traverse('preorder')))
print('Nodes in postorder: ', ', '.join(node.name for node in t.traverse('postorder')))

We can also iterate over the tree itself. This will be interpreted as iterating over the *leaves* of the tree:

In [ ]:
print('Leaves:', ', '.join(leaf.name for leaf in t))

Or, more explicitly, instead of `for leaf in t` we can also use `for leaf in t.leaves()`.

To only iterate over the descendants (that is, excluding the root node itself), instead of `t.traverse()` we can use `t.descendants()`. And to iterate over the ancestors, we can use `t.ancestors()`:

In [ ]:
print('All ancestors of node C:', ', '.join(node.name for node in t['C'].ancestors()))

This covers the basics of traversing trees, but there is more information in the [corresponding section of ETE's tutorial](https://etetoolkit.github.io/ete/tutorial/tutorial_trees.html#traversing-browsing-trees).

<div style="background-color:lightblue; padding:1em">

  ### Exercise: Traversing nodes
  
ETE's default traversal method, "levelorder", visits nodes from closer to farther from the root.

- Check that as we are traversing the nodes, all the ancestors of the current node have already been visited
- Is it true too if we traverse in preorder? And in postorder?
</div>

---

## Calculating distances between nodes: branch lengths and topology

Let's create a tree with branch lengths:

In [ ]:
t = Tree('(((A:1,B:2):2,C:3):1,((D:0.3,E:0.8):1,F:0.4):3);')
print(t)

Similar to what we saw before, we could show the branch lengths (`dist`) with the appropriate arguments to `t.to_str()`:

In [ ]:
print(t.to_str(props=['dist', 'name']))

If we want to know the **distance** between two nodes, we can use the `get_distance()` method:

In [ ]:
print('The distance between A and C is', t.get_distance('A', 'C'))

In addition to the sum of distances to get from one node to the other, we can get the *topological distance*, that is, the number of nodes between them. For that we can use the argument `topological=True`:

In [ ]:
print('The number of nodes between A and F is', t.get_distance('A', 'F', topological=True))

We can use the method `get_farthest_node()` to get the most distant node. Alternatively, `get_farthest_leaf()` will return the most distant descendant (always a leaf). If more than one node matches the farthest distance, the first found by the function is returned.

And we can use the method `get_midpoint_outgroup()` to get the node that partitions the tree into two balanced branches in terms of node distances:

In [ ]:
t = Tree('((A:0.5,B:0.5)AB:0.5,C:1,(D:1,(E:1,F:2)EF:0.5)DEF:0.5);', parser=1)

print(t.to_str(props=['dist', 'name']))

In [ ]:
midpoint = t.get_midpoint_outgroup()

print('Middle point is at node:', midpoint.name)

Sometimes we want to obtain a balanced rooting of the tree. We will see later how to reroot (or "set an outgroup") in any node. If we set as an outgroup the midpoint, we will have a balanced tree.

---

### Manipulating trees: add, remove, detach, prune, reroot

ETE provides operations for tree manipulation. They make some basic checks and take care of automatically setting the proper values for `up` and `children` of the nodes involved. We will now look at the most common ones.

### Adding

Let's start with two separate trees:

In [ ]:
t1 = Tree('(((A,B),C),D);')
t2 = Tree('(E,F);')
print(t1)
print(t2)

We can create a tree *concatenating*, or *adding*, the previous trees with `+`:

In [ ]:
t = t1 + t2

In [ ]:
print(t)

Or *add new children* by using the `add_child()` method:

In [ ]:
t['E'].add_child(name='X')
t['E'].add_child(name='Y')
print(t.to_str(props=['name'], compact=True))

The same `add_child()` method, when used on an existing tree, can also be used to *append* it:

In [ ]:
subtree = Tree('(N1,N2);')
print('New subtree that we will attach:')
print(subtree)

t['F'].add_child(subtree)
print('Result of attaching the subtree to node F:')
print(t.to_str(props=['name'], compact=True))

If we wanted to add several children at the same time, they can be sent as a list to the `add_children()` method.

Another way of adding is by using `add_sister()` to add at the same level:

In [ ]:
t['C'].add_sister(name='S')
print(t.to_str(props=['name'], compact=True))

### Removing

The two most basic ways of removing a node are with `remove_child()` and `detach()`. For example, we can remove E (the parent of X and Y) from its parent by doing:

In [ ]:
t['E'].up.remove_child(t['E'])
print(t.to_str(props=['name'], compact=True))

We see that it also removed the X and Y nodes with it, since they were hanging on E.

We can do the same for the parent of N1 (and N2) in a more direct way with:

In [ ]:
t['N1'].up.detach()
print(t.to_str(props=['name'], compact=True))

If instead of removing/detaching, we `delete()` a node, the children of the deleted node are kept. For example, if we remove A's parent:

In [ ]:
t['A'].up.delete()
print(t.to_str(props=['name'], compact=True))

We can see that A stays in the tree, and has been transferred (with B) to the old parent.

Finally, we can also `prune()` the tree in order to *keep only* some leaf nodes:

In [ ]:
t.prune(['A', 'C', 'D', 'F'])
print(t.to_str(props=['name'], compact=True))

As usual, we can find more examples and detailed information in [ETE's documentation](https://etetoolkit.github.io/ete/tutorial/tutorial_trees.html#how-to-delete-eliminate-or-remove-detach-nodes).

### Rerooting

Another common tree manipulation consists of changing the root to a different location in the tree. In phylogenetics this is a crucial step prior to the interpretation of trees, since it will determine the evolutionary relationships among the species involved.

Let's start with a tree where we name all the nodes, so we can follow more easily:

In [ ]:
t = Tree('((A,B)C,(D,((E,F)G,H)I)J)root;', parser=1)
print(t.to_str(props=['name']))

By calling `set_ougroup()` we can set G as the outgroup. It means that we change the position of the root of the tree, so that G representes one of the main tree branches coming from it.

In [ ]:
t.set_outgroup('G')
print(t.to_str(props=['name']))

---

## Performance tricks

### Caching leaves

Sometimes we want to access the content of different nodes very frequently. Traversing the tree to get the leaves of each node over and over would produce significant slowdowns in certain algorithms. For those cases, ETE provides a convenient method to cache frequently used data.

For example, let's start with this tree:

In [ ]:
t = Tree('((((A,B)G,C)H,D)I,(E,F)J)root;', parser=1)
print(t.to_str(props=['name']))

Let's say we want to know, for all the nodes, how many leaves hang from them.

We could traverse the tree and, at each node, do something like `len(list(node.leaves()))`. But we would be traversing the tree from that node to find them, when we are already inside a tree traversal. The complexity (and time) for those operations would increase quadratically, making them too slow for big trees.

Instead, we can use the method `get_cached_content()`. It returns a dictionary in which keys are node instances and values represent the content of such nodes (by default, "content" is understood as a set of leaf nodes, but it can be set to some property too instead).

After we retrieve this cached data, looking up the number of leaves under a node (or their names, or anything similar) will be instantaneous:

In [ ]:
leaves = t.get_cached_content()

# Print the size of each node, without the need of traversing the subtrees every time.
for node in t.traverse():
    print('Node %s has %d leaves.' % (node.name, len(leaves[node])))

### Traversing in prepostorder

In some complex algorithms we want to perform certain operations on the nodes of the tree while we are descending through the nodes, and different ones when we are going back up.

In addition to the traversals that we have seen, ETE also provides a pre+post order traversal, `iter_prepostorder()`, where nodes are visited both when descending and when ascending. It yields the nodes in the order that are visited, and a flag that says whether the node has already been visited.

![levelorder2](440px-Sorted_binary_tree_ALL_RGB.svg.png)

<!-- from https://upload.wikimedia.org/wikipedia/commons/thumb/7/75/Sorted_binary_tree_ALL_RGB.svg/440px-Sorted_binary_tree_ALL_RGB.svg.png -->

**prepostorder** (in order of touching the <span style="color:red">red</span> *and* <span style="color:blue">blue</span> dots: F, B, A, D, C, E, D, B, G, I, H, I, G, F)

In [ ]:
t = Tree('((A,(C,E)D)B,((H)I)G)F;', parser=1)

print(t.to_str(props=['name']))

print('Nodes in prepostorder (with * if the node has already been visited):')
print(', '.join(node.name + ('*' if visited else '') for visited, node in t.iter_prepostorder()))

For example, ETE uses this strategy when drawing large trees, which we will use soon. This allows it to gather efficiently different kinds of information about what to display.

---

## Basic graphical visualization

Let's create a tree with 100 leaves and random branch lengths:

In [ ]:
from random import random

t = Tree()  # create an empty tree

t.populate(100, dist_fn=random)  # populate it randomly, and with random branch lengths

We can try to see it as text, but it's going to be messy:

In [ ]:
print(t)

We can access a much nicer interface to explore the tree with the following command:

In [ ]:
t.explore()

And from this point on, we can interact with the tree in both the graphical explorer and the console at the same time.

Later on, we will see how to customize the visualization to show all the information that we want, by setting different styles to nodes and adding textual and graphical pieces.

<div style="background-color:lightblue; padding:1em">

  ### Exercise: Exploring trees

- Open a tree and explore it, or use the control panel to upload a file or an example
- Move around the tree with the mouse, clicking and dragging, and zooming with the wheel
- Play with the options in the control panel and the context menu (*right click*)
</div>